### PREREQUISITES

#### Importing Modules and Tables

In [1]:
import pandas as pd

In [26]:
candidate_df = pd.read_csv(r"C:\Users\SalauddinKhan\Desktop\PYTHON\candidates_cht2024.csv")
vacancy_df = pd.read_csv(r"C:\Users\SalauddinKhan\Desktop\PYTHON\vacancy_table_cht2024.csv")

In [27]:
'''def func(string):
    char_map = {}
    for i, s in enumerate(string):
        if s in char_map:
            return i
        char_map[s] = i
    return len(string)'''
print()

In [28]:
'''ef is_valid_string(s): 
    stack = [] 
    mapping = {')': '(', '}': '{', ']': '['} 
    for char in s: 
        if char in mapping:
            top_element = stack.pop() if stack else '#'
            if mapping[char] != top_element: 
                return False
        else: 
            stack.append(char)
    return not stack 
is_valid_string("([{(})])") #-> False'''
print()

In [5]:
'''def fnrc(s):
    temp = s
    for i, c in enumerate(s):
        if c in s[i+1:]:
            
            continue
        else:
            return i
    return -1

fnrc("lelomomei")'''
print()

#### ARC Table from Notice

In [29]:
arc_dict = {'arc_code': ['1', '2', '3', '4', '5', '6', '8', '9'], 
            'arc_year': [5, 3, 10, 13, 15, 3, 3, 8],
            'arc_cat1': ['1,2', '6', '0,9', '6', '1,2', '0,1,2,6,9', '0,6,9', '1,2'],
            'arc_cat2': ['', '', '', '', '', '3', '', ''],
            'arc_cat3': ['', '', '4,5,8', '4,5,8', '4,5,8', '', '', ''],
            'arc_gender': ['1,2,3', '1,2,3', '1,2,3', '1,2,3', '1,2,3', '1,2,3', '1,2,3', '1,2,3'],
            'arc_year_against_ur': [ None, None, 10, 10, 10, None, 3, 3]
           }

arc_df = pd.DataFrame(arc_dict)

#### Cutoff Table from Notice

In [30]:
cutoff_data = [
    {'category': 0, 'paper1': 131.25, 'paper2': 50},
    {'category': 1, 'paper1': 126.75, 'paper2': 40},
    {'category': 2, 'paper1': 96.50, 'paper2': 40},
    {'category': 3, 'paper1': 999.99, 'paper2': 999.99},
    {'category': 4, 'paper1': 115.50, 'paper2': 40},
    {'category': 5, 'paper1': 75.75, 'paper2': 40},
    {'category': 6, 'paper1': 150.75, 'paper2': 50},
    {'category': 7, 'paper1': 42.0, 'paper2': 40},
    {'category': 8, 'paper1': 40, 'paper2': 40},
    {'category': 9, 'paper1': 150.75, 'paper2': 60},
]
cutoff_df = pd.DataFrame(cutoff_data)

In [31]:
candidate_df['dob'] = pd.to_datetime(candidate_df['dob'])

### MERIT

#### CODE

In [32]:
def get_merit(candidate_df):    
    
    candidate_df['merit'] = None
    
    candidates =  candidate_df[
    (candidate_df['total'].notnull()) &
    (candidate_df['rejection_provision'].astype(str).isin(('C','W')))
    ].sort_values(by = ['total', 'paper2', 'gen_hindi', 'dob', 'cand_name'], ascending=[False, False, False, True, True])
    
    candidates['merit'] = range(1, len(candidates) + 1)

    rollno_to_merit = candidates.set_index('rollno')['merit'].to_dict()
    candidate_df['merit'] = candidate_df['rollno'].map(rollno_to_merit)
    
    return "{flag_name} updated successfully for {merit} candidates".format(
        flag_name='merit',
        merit=len(candidates))

#### EXECUTION

In [33]:
get_merit(candidate_df)

'merit updated successfully for 1940 candidates'

In [34]:
candidate_df[['rollno', 'merit']].head()

,rollno,merit
0,1004000027,785
1,1004000097,825
2,1004000121,1429
3,1004000130,678
4,1403000009,1347


### CUTOFF FLAG

#### CODE

In [35]:
def get_cutoff(cutoff_df, candidate_df):

    cutoff = 'cutoff_flag'
    candidate_df[cutoff] = None
    cutoff_dict = {}

    for index, row in cutoff_df.iterrows():
        category = int(row['category'])
        paper1 = float(row['paper1'])
        paper2 = float(row['paper2'])
        cutoff_dict[category] = [paper1, paper2]
    
    sorted_candidates = candidate_df[candidate_df['merit'].notnull()].sort_values(by='merit')

    def calculate_cutoff(candidate):
        cutoff = ''

        if (candidate['paper1'] >= cutoff_dict[9][0]) & (candidate['paper2'] >= cutoff_dict[9][1]):
            cutoff += '9'

        for cat in [0, 1, 2, 6]:
            if (candidate['cat1'] == cat) & (cat in cutoff_dict):
                if (candidate['paper1'] >= cutoff_dict[cat][0]) & (candidate['paper2'] >= cutoff_dict[cat][1]):
                    cutoff += str(cat)

        if (candidate['cat2'] == 3) & (3 in cutoff_dict):
            if (candidate['paper1'] >= cutoff_dict[3][0]) & (candidate['paper2'] >= cutoff_dict[3][1]):
                cutoff += '3'

        for cat in [4, 5, 7, 8]:
            if (candidate['cat3'] == cat) & (cat in cutoff_dict):
                if (candidate['paper1'] >= cutoff_dict[cat][0]) & (candidate['paper2'] >= cutoff_dict[cat][1]):
                    cutoff += str(cat)

        return cutoff

    sorted_candidates[cutoff] = sorted_candidates.apply(lambda row: calculate_cutoff(row), axis=1)
    candidate_df.update(sorted_candidates)

    return f"{cutoff} updated successfully"


#### EXECUTION

In [36]:
get_cutoff(cutoff_df, candidate_df)

'cutoff_flag updated successfully'

In [37]:
candidate_df[['rollno', 'cutoff_flag']].head()

,rollno,cutoff_flag
0,1004000027,91
1,1004000097,9
2,1004000121,2
3,1004000130,90
4,1403000009,91


In [38]:
candidate_df['cutoff_flag'].unique()

array(['91', '9', '2', '90', '1', '0', '96', '4', '5', '', '92', '6',
       '14', '7', '914', '8', '98', '94', '904', '964', '04', '965', '17'],
      dtype=object)

In [102]:
cand['cutoff_flag'].value_counts()

cutoff_flag
9      500
96     470
1      238
90     163
91     155
2      139
0      124
        46
4       24
6       18
92      16
7       15
5       10
964      5
8        4
94       4
14       2
904      2
914      1
98       1
04       1
965      1
17       1
Name: count, dtype: int64

### DOB FLAG

#### PREREQIUSITES

In [39]:
mask = candidate_df['cat1'].isnull()
candidate_df.loc[mask, 'cat1'] = ''
candidate_df.loc[~mask, 'cat1'] = candidate_df.loc[~mask, 'cat1'].astype(int).astype(str)

mask = candidate_df['cat2'].isnull()
candidate_df.loc[mask, 'cat2'] = ''
candidate_df.loc[~mask, 'cat2'] = candidate_df.loc[~mask, 'cat2'].astype(int).astype(str)

mask = candidate_df['cat3'].isnull()
candidate_df.loc[mask, 'cat3'] = ''
candidate_df.loc[~mask, 'cat3'] = candidate_df.loc[~mask, 'cat3'].astype(int).astype(str)

mask = candidate_df['agerelax_code'].isnull()
candidate_df.loc[mask, 'agerelax_code'] = ''
candidate_df.loc[~mask, 'agerelax_code'] = candidate_df.loc[~mask, 'agerelax_code'].astype(int).astype(str)

C:\Users\SalauddinKhan\AppData\Local\Temp\ipykernel_2660\1307339542.py:2: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  candidate_df.loc[mask, 'cat1'] = ''
C:\Users\SalauddinKhan\AppData\Local\Temp\ipykernel_2660\1307339542.py:6: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  candidate_df.loc[mask, 'cat2'] = ''
C:\Users\SalauddinKhan\AppData\Local\Temp\ipykernel_2660\1307339542.py:10: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  candidate_df.loc[mask, 'cat3'] = ''
C:\Us

In [40]:
candidate_df[['exsm_yrs', 'exsm_months', 'exsm_days']] = candidate_df['service_period'].str.extract(r'(\d+) Year[s]* (\d+) Month[s]* (\d+) Day[s]*')

candidate_df[['exsm_yrs', 'exsm_months', 'exsm_days']] = candidate_df[['exsm_yrs', 'exsm_months', 'exsm_days']].apply(pd.to_numeric)

#### CODE

In [41]:
def get_dob_flag(candidate_df, arc_df):
    dob_to_date = pd.to_datetime('08/01/2006', format="%m/%d/%Y")
    dob_from_date = pd.to_datetime('08/02/1994', format="%m/%d/%Y")
    
    arc_year_dict = dict(zip(arc_df['arc_code'].astype(str), arc_df['arc_year']))

    candidate_df['dob_flag'] = ''

    def calculate_verification_flag(row):
        dob = row['dob']
        verification_flag = ''
        
        if dob > dob_to_date:
            verification_flag = 'U'
        elif dob_from_date <= dob <= dob_to_date:
            verification_flag = '9'
        elif dob < dob_from_date:
            agerelax_code = row['agerelax_code']
            
            if pd.isna(agerelax_code) or agerelax_code == '':
                agerelax_code = '99'
            
            if isinstance(agerelax_code, str) and len(agerelax_code.strip()) == 1:
                agerelax_code = agerelax_code.strip()
            
            age_rlx_year = arc_year_dict.get(agerelax_code, 0)
            
            if agerelax_code == '6':
                ex_service_years = row['exsm_yrs']
                ex_service_months = row['exsm_months']
                ex_service_days = row['exsm_days']
                age_rlx_year += int(ex_service_years) if ex_service_years else 0
                new_dob = dob + pd.DateOffset(years=age_rlx_year, months=ex_service_months, days=ex_service_days)
            
            else:
                extract_year = dob.year
                extract_month = dob.month
                extract_day = dob.day
                new_year = extract_year + int(age_rlx_year)
                
                if extract_month == 2 and extract_day == 29:
                    extract_day = 28
            
                new_dob = pd.to_datetime(f"{new_year}-{extract_month}-{extract_day}", format='%Y-%m-%d')
                
            if new_dob > dob_from_date:
                if len(agerelax_code) < 2:
                    verification_flag = '0' + agerelax_code
                else:
                    verification_flag = agerelax_code
            else:
                verification_flag = '99'

        return verification_flag

    sorted_candidates = candidate_df[(candidate_df['merit'].notnull())]
    
    sorted_candidates['dob_flag'] = sorted_candidates.apply(calculate_verification_flag, axis=1)

    candidate_df.update(sorted_candidates)
    
    return f"dob_flag updated successfully"

#### EXECUTION

In [42]:
get_dob_flag(candidate_df, arc_df)

'dob_flag updated successfully'

In [43]:
candidate_df[['rollno', 'dob_flag']].head()

,rollno,dob_flag
0,1004000027,01
1,1004000097,9
2,1004000121,01
3,1004000130,9
4,1403000009,9


In [44]:
candidate_df['dob_flag'].unique()

array(['01', '9', '02', '03', '06', '05', '99', '04'], dtype=object)

In [99]:
cand['dob_flag'].value_counts()

dob_flag
9     1533
01     217
02      94
06      57
04      15
03      14
05       9
99       1
Name: count, dtype: int64

In [52]:
overage = candidate_df[
    candidate_df['dob_flag'] == '01'
][['rollno','dob', 'cat1', "cat2", "cat3","merit", "agerelax_code","cutoff_flag"]].head(10)


print(overage)

        rollno        dob cat1 cat2 cat3  merit agerelax_code cutoff_flag
0   1004000027 1991-01-27    1              785             1          91
2   1004000121 1993-07-29    2             1429             1           2
11  1403000057 1993-07-27    1              592             1          91
16  1403000112 1990-02-27    1             1566             1           1
17  1403000127 1992-08-01    1             1346             1           1
18  1403000128 1990-10-11    1             1360             1           1
29  1403000263 1992-12-30    1              491             1          91
32  1403000289 1993-10-28    1              467             1          91
57  2002000023 1989-11-20    1             1230             1          91
72  2006000004 1991-05-05    2             1924             1           2


In [55]:
overage = candidate_df[
    candidate_df['dob'] == '1994-08-02'
][['rollno','dob', 'cat1', "cat2", "cat3","merit", "agerelax_code","cutoff_flag",'dob_flag']].head(10)


print(overage)

          rollno        dob cat1 cat2 cat3  merit agerelax_code cutoff_flag  \
1908  9001000139 1994-08-02    9              848                         9   

     dob_flag  
1908        9  


### CATSEL DOB FLAG

#### PREREQUISITES

In [56]:
arc_df['arc_code'] = arc_df['arc_code'].astype(str)

mask = arc_df['arc_cat2'].isnull()
arc_df.loc[mask, 'arc_cat2'] = ''

mask = arc_df['arc_cat3'].isnull()
arc_df.loc[mask, 'arc_cat3'] = ''

In [57]:
arc_df['arc_code'] = arc_df['arc_code'].apply(lambda x: str(x).zfill(2))

#### CODE

In [58]:
def get_catsel_dob(candidate_df, arc_df):
    dob_from_date = pd.to_datetime('08/02/1994', format="%m/%d/%Y")
    
    dob_flag = 'dob_flag'
    catsel_dob_flag = 'catsel_dob_flag'

    candidate_df[catsel_dob_flag] = ''
    
    filtered_and_sorted_candidates = candidate_df[(candidate_df['merit'].notnull()) 
    & (candidate_df[dob_flag].notnull()) 
    & (candidate_df[dob_flag] != '')]
    
    def calculate_catsel_dob(row, dob_flag_name):
        dob_flag = row[dob_flag_name]
       
        if dob_flag is not None and dob_flag != '' and dob_flag != '9':
            matching_rows = arc_df[arc_df['arc_code'] == dob_flag]
            if not matching_rows.empty:
                catsel = matching_rows.iloc[0]
                catsel_cat1 = catsel['arc_cat1'].split(',')
                catsel_cat2 = catsel['arc_cat2'].split(',')
                catsel_cat3 = catsel['arc_cat3'].split(',')
                catsel_gender = catsel['arc_gender'].split(',')
                condition = (str(row['cat1']) in catsel_cat1) & (str(int(row['gender'])) in catsel_gender)
                catsel_dob = ''
                if row['cat2'] == '3' and row['cat2'] in catsel_cat2:
                    catsel_dob = row['cat2']
                else:
                    if pd.isnull(catsel['arc_year_against_ur']):
                        catsel['arc_year_against_ur'] = 0
                    dob_datetime = pd.to_datetime(row['dob'])
                    extract_year = dob_datetime.year
                    extract_month = dob_datetime.month
                    extract_day = dob_datetime.day
                    if (extract_day == 29) and (extract_month == 2):
                        extract_day = 28
                    new_year = extract_year + int(catsel['arc_year_against_ur'])
                    new_dob = pd.to_datetime(f'{new_year}-{extract_month}-{extract_day}')
                    catsel_dob_decision = new_dob > dob_from_date
                    if (str(row['cat3']) in catsel_cat3 and str(row['cat3']) != ''):
                        if catsel_dob_decision:
                            catsel_dob = '9'
                        else:
                            catsel_dob = row['cat3']  
                    else:
                        catsel_dob = '9' if catsel_dob_decision else row['cat1']
                return catsel_dob if condition else ''
            else:
                return ''
        else:
            return dob_flag
    
    filtered_and_sorted_candidates[catsel_dob_flag] = filtered_and_sorted_candidates.apply(calculate_catsel_dob, args = (dob_flag, ), axis=1)
    candidate_df.update(filtered_and_sorted_candidates)
    
    print('Successfully updated ' + catsel_dob_flag)

#### EXECUTION

In [59]:
get_catsel_dob(candidate_df, arc_df)

Successfully updated catsel_dob_flag


In [60]:
candidate_df[['rollno', 'catsel_dob_flag']].head()

,rollno,catsel_dob_flag
0,1004000027,1
1,1004000097,9
2,1004000121,2
3,1004000130,9
4,1403000009,9


In [100]:
cand['catsel_dob_flag'].value_counts()

catsel_dob_flag
9    1568
1     162
6      94
3      57
2      55
4       3
        1
Name: count, dtype: int64

In [65]:
overage = candidate_df[
    candidate_df['dob_flag'] == '99'
][['rollno','dob', 'cat1', "cat2", "cat3","merit", "agerelax_code","cutoff_flag",'dob_flag','catsel_dob_flag']].head(10)


print(overage)

         rollno        dob cat1 cat2 cat3  merit agerelax_code cutoff_flag  \
375  2201001170 1987-08-25    6    3        1201             2          96   

    dob_flag catsel_dob_flag  
375       99                  


### CATSEL FLAG

#### CODE

In [66]:
def get_catsel(candidate_df):
    cutoff_flag = 'cutoff_flag'
    catsel_flag = 'catsel'
    candidate_df[catsel_flag] = ''
    
    candidates = candidate_df[candidate_df['merit'].notnull()].sort_values(by='rollno')
    
    def update_cutoff_flag(candidate_record):
        if candidate_record['cat2'] == '3' and candidate_record['exs_reservation'] == 'No':
            candidate_record[cutoff_flag] = candidate_record[cutoff_flag].replace('3', '').strip()

        if candidate_record['catsel_dob_flag'] in ['1', '2', '6', '4', '5', '7', '8']:
            return candidate_record[cutoff_flag].replace('9', '').strip()
        elif pd.isna(candidate_record['catsel_dob_flag']) or candidate_record['catsel_dob_flag'] == '':
            return ''
        else:
            return candidate_record[cutoff_flag]
    
    candidate_df[catsel_flag] = candidates.apply(update_cutoff_flag, axis=1)

    return 'Successfully updated '+ catsel_flag

#### EXECUTION

In [67]:
get_catsel(candidate_df)

'Successfully updated catsel'

In [68]:
candidate_df[['rollno', 'catsel']].head()

,rollno,catsel
0,1004000027,1
1,1004000097,9
2,1004000121,2
3,1004000130,90
4,1403000009,91


In [101]:
cand['catsel'].value_counts()

catsel
9      500
96     383
1      296
90     163
2      144
0      124
6      104
91      97
        47
4       24
7       15
92      11
5       10
8        4
94       4
964      4
14       3
904      2
98       1
04       1
965      1
17       1
64       1
Name: count, dtype: int64

In [73]:
overage = candidate_df[
    candidate_df['cutoff_flag'] == '96'
][['rollno',"cutoff_flag",'dob_flag','catsel_dob_flag','exs_reservation','catsel']].head(10)


print(overage)

        rollno cutoff_flag dob_flag catsel_dob_flag exs_reservation catsel
10  1403000047          96       02               6             NaN      6
14  1403000107          96        9               9             NaN     96
43  1403000428          96        9               9             NaN     96
45  1403000474          96       06               3             Yes     96
50  1403000512          96       02               6             NaN      6
52  1403000552          96        9               9             NaN     96
53  1403000553          96        9               9             NaN     96
55  2002000009          96       02               6             NaN      6
60  2002000041          96        9               9             NaN     96
61  2002000042          96        9               9             NaN     96


### ALLOCATION

#### PREREQUISITES

In [84]:
cand = candidate_df.copy()
vac = vacancy_df.copy()

In [85]:
cand[['allocated_category', 'allocated_post', 'allocated_dept', 'allocated_against_ur']] = None

In [86]:
mask = cand['post_pref'].isnull()
cand.loc[mask, 'post_pref'] = ''
cand.loc[~mask, 'post_pref'] = cand.loc[~mask, 'post_pref'].astype(str)

mask = cand['catsel'].isnull()
cand.loc[mask, 'catsel'] = ''
cand.loc[~mask, 'catsel'] = cand.loc[~mask, 'catsel'].astype(str)

mask = cand['catsel_dob_flag'].isnull()
cand.loc[mask, 'catsel_dob_flag'] = ''
cand.loc[~mask, 'catsel_dob_flag'] = cand.loc[~mask, 'catsel_dob_flag'].astype(str)

In [87]:
vac['allocated_hc'] = 0
vac['allocated_hc_prev'] = 0

In [88]:
vac['left_vacancy'] = vac['current']
vacancy_dict = {}
vac['key'] = vac['category_code'].astype(str) + vac['dept_code'].astype(str)

for index, row in vac.iterrows():
    key = row['key']
    vacancy_dict[key] = row.to_dict()

#### CODE

In [89]:
def update_allocation(candidates_df, category, post, department, allocated_against_ur, candidate, vacancy_dict):
    if str(category) in ['3', '4', '5', '7', '8']:
    
        key_cat1 = str(candidate['cat1']) + str(department)
        
        if (key_cat1 not in vacancy_dict.keys()) or (vacancy_dict[key_cat1]['initial_vacancy'] == 0):
            keyCat2= '9' + str(department)
            
            if (keyCat2 not in vacancy_dict.keys()) or (vacancy_dict[keyCat2]['initial_vacancy'] == 0):
                return False
            else:
                if vacancy_dict[keyCat2]['allocated_hc'] != vacancy_dict[keyCat2]['initial_vacancy']:
                    vacancy_dict[keyCat2]['allocated_hc'] += 1
                allocated_against_ur = '1'
        else:
            if vacancy_dict[key_cat1]['allocated_hc'] != vacancy_dict[key_cat1]['initial_vacancy']:
                vacancy_dict[key_cat1]['allocated_hc'] += 1
            else :
                keyCat2= '9' + str(department)
                if candidate['catsel_dob_flag'] == '9' and keyCat2 in vacancy_dict.keys():
                    if vacancy_dict[keyCat2]['allocated_hc'] != vacancy_dict[keyCat2]['initial_vacancy']:
                        vacancy_dict[keyCat2]['allocated_hc'] += 1
                        allocated_against_ur = '1'
                    else:
                        return False      
                else:
                    return False
                        
    candidates_df.loc[candidate.name, 'allocated_category'] = category
    candidates_df.loc[candidate.name, 'allocated_post'] = post
    candidates_df.loc[candidate.name, 'allocated_dept'] = department
    candidates_df.loc[candidate.name, 'allocated_against_ur'] = allocated_against_ur
    
    return True

In [90]:
def allocate_candidates(candidates_df, vacancy_dict):
     
    filtered_candidates = candidates_df[(candidates_df['merit'].notnull())].sort_values(by='merit')
    
    for idx, candidate in filtered_candidates.iterrows():
        allocated = False
        roll = candidate['rollno']
        department_preference = candidate['post_pref']
        DOB = candidate['dob']
        
        for department in department_preference.split(','):

            catsel = candidate['catsel']
            
            for category in catsel:
                key =  str(category) + str(department)
                
                allocated_against_ur = ''
                
                if key in vacancy_dict and vacancy_dict[key]['current'] > 0:
                    post = vacancy_dict[key]['dept_code']
                                
                    allocated = update_allocation(candidates_df, category, post, department, allocated_against_ur, candidate, vacancy_dict)
                    if allocated:
                        vacancy_dict[key]['current'] -= 1
                        vacancy_dict[key]['allocated'] += 1
                        break
            if allocated:
                break

    return vacancy_dict

In [91]:
def adjust_vacancy(candidates_df, vacancy_df):
    vacancy_df['current'] = vacancy_df['initial_vacancy'] - vacancy_df['allocated_hc']
    vacancy_df['allocated'] = 0
    vacancy_df['left_vacancy'] = 0
        
    vacancy_df['allocated_hc_prev'] = vacancy_df['allocated_hc']
    vacancy_df['allocated_hc'] = 0

    return vacancy_df

In [92]:
def find_lowest_marks(candidates_df, vacancy_df):
    vc = []
    vacancy_df['min_total'] = 0
    for _, row in vacancy_df.iterrows():
        lmv = {
            "dept_code": str(row["dept_code"]),
            "category_code": str(row["category_code"])
        }

        vc.append(lmv)

    print("Vector size--->", len(vc))

    for lmv1 in vc:
        highest_merit_candidate = get_highest_merit_candidate(candidates_df, lmv1["dept_code"], lmv1["category_code"])
        update_vacancy_table(vacancy_df, lmv1["dept_code"], lmv1["category_code"], highest_merit_candidate)

def get_highest_merit_candidate(candidates_df, department_code, category_code):
    highest_merit_candidate = None

    mask = (
        (candidates_df["allocated_post"] == str(department_code))
        & (candidates_df["allocated_category"] == str(category_code))
    )

    filtered_cand = candidates_df[mask]

    if not filtered_cand.empty:
        highest_merit_candidate = filtered_cand.loc[filtered_cand['merit'].idxmax()]
    
    return highest_merit_candidate

def update_vacancy_table(vacancy_df, department_code, category_code, highest_merit_candidate):
    key = str(category_code) + str(department_code)
    mask = (vacancy_df['key'] == key)
    vacancy = vacancy_df.loc[mask].copy()

    if highest_merit_candidate is not None:
        vacancy["min_marks"] = highest_merit_candidate['total']
        vacancy["min_paper2"] = highest_merit_candidate['paper2']
        vacancy["min_marks_cand_dob"] = highest_merit_candidate['dob']
        vacancy["min_part1_gen_hindi"] = highest_merit_candidate['gen_hindi']
        
        vacancy_df.loc[mask, 'min_marks'] = vacancy["min_marks"]
        vacancy_df.loc[mask, 'min_paper2'] = vacancy["min_paper2"]
        vacancy_df.loc[mask, 'min_marks_cand_dob'] = vacancy["min_marks_cand_dob"]
        vacancy_df.loc[mask, 'min_part1_gen_hindi'] = vacancy["min_part1_gen_hindi"]


#### EXECUTION

##### FIRST TIME

In [93]:
upd_vacancy_dict = allocate_candidates(cand, vacancy_dict)

##### ADJUST LOOP

In [94]:
updated_vacancy_df = pd.DataFrame.from_dict(upd_vacancy_dict, orient='index')
i =0
while (updated_vacancy_df['allocated_hc_prev'] != updated_vacancy_df['allocated_hc']).any():
    i+=1
    print('Attempt-'+str(i))
    upd_vacancy_df = adjust_vacancy(cand, updated_vacancy_df)
    
    upd_vacancy_df['left_vacancy'] = upd_vacancy_df['current']
    vacancy_dict = {}
    upd_vacancy_df['key'] = upd_vacancy_df['category_code'].astype(str) + upd_vacancy_df['dept_code'].astype(str)
    
    for index, row in upd_vacancy_df.iterrows():
        key = row['key']
        vacancy_dict[key] = row.to_dict()
    cand[['allocated_category', 'allocated_post', 'allocated_dept', 'allocated_against_ur']] = None
    upd_vacancy_dict = allocate_candidates(cand, vacancy_dict)
    updated_vacancy_df = pd.DataFrame.from_dict(upd_vacancy_dict, orient='index')

Attempt-1


In [95]:
cand[cand['allocated_category'].notnull()].shape[0]

272

In [96]:
updated_vacancy_df['left_vacancy'] = updated_vacancy_df['current']
updated_vacancy_df['current'] = updated_vacancy_df['left_vacancy'] + updated_vacancy_df['allocated']

In [97]:
find_lowest_marks(cand, updated_vacancy_df)

Vector size---> 234


### WRITING TO CSV

In [98]:
cand.to_excel(r"cht2024_with_allocation.xlsx", index=False, engine='xlsxwriter')
updated_vacancy_df.to_excel(r"vacancy.xlsx", index=False, engine='xlsxwriter')

In [40]:
cand.to_csv(r"C:\Users\Aviral Chaudhary\Downloads\RP_JHT_2023\allocated_candidates.csv", index = False)
cand[cand['allocated_category'].notnull()].to_csv(r"C:\Users\Aviral Chaudhary\Downloads\RP_JHT_2023\only_allocated_candidates.csv", index = False)
updated_vacancy_df.to_csv(r"C:\Users\Aviral Chaudhary\Downloads\RP_JHT_2023\allocated_vacancy.csv", index = False)